# PyWry Toolbar System Demo

This notebook demonstrates PyWry's powerful toolbar system, from basic usage to advanced features.

## What You'll Learn

1. **Basic Toolbar** - Adding buttons and handling clicks
2. **Input Components** - Text, number, select, and slider inputs with 2-way binding
3. **Multiple Toolbars** - Using different positions (top, bottom, left, right, inside)
4. **Div Containers** - Custom HTML content within toolbars
5. **Collapsible & Resizable** - Interactive toolbar features
6. **Advanced Example** - A complete interactive dashboard

## Prerequisites

- `pywry` library installed
- `anywidget` (optional, for notebook inline rendering)

In [ ]:
# Enable debug mode
import os
os.environ["PYWRY_DEBUG"] = "1"

# Core imports from local pywry
from pywry import (
    PyWry,
    Toolbar,
    Button,
    Select,
    MultiSelect,
    TextInput,
    NumberInput,
    SliderInput,
    DateInput,
    RangeInput,
    Toggle,
    Checkbox,
    RadioGroup,
    TabGroup,
    Option,
    Div,
)


app = PyWry()

---

## 1. Basic Toolbar - Button Click Handler

The simplest toolbar contains a button that triggers a Python callback. When clicked:
1. JavaScript emits an event with the button's `event` name
2. Python callback receives the event data
3. Python can respond by sending JavaScript back via `widget.emit()`

In [ ]:
# State for tracking clicks
click_count = 0

# Create a simple toolbar with one button
# Note: "neutral" is the blue accent button for primary actions
basic_toolbar = Toolbar(
    position="top",
    items=[
        Button(label="Click Me!", event="app:click", variant="neutral"),
        Button(label="Reset", event="app:reset", variant="primary"),
    ],
)

# HTML content with a counter display
basic_html = """
<div style="text-align: center; overflow: hidden;">
    <h1 id="counter" style="font-size: 48px; margin: 10;">0</h1>
    <p style="color: var(--pywry-text-secondary);">Click the button to increment</p>
</div>

<script>
// Listen for count updates from Python
if (window.pywry && window.pywry.on) {
    window.pywry.on('app:update_count', function(data) {
        document.getElementById('counter').textContent = data.count;
    });
    window.pywry.on('app:reset_count', function(data) {
        document.getElementById('counter').textContent = '0';
    });
}
</script>
"""

def on_button_click(data, event_type, label):
    """Handle button click - update the displayed count."""
    print(f"[CALLBACK] on_button_click called! data={data}, event_type={event_type}")
    global click_count
    click_count += 1
    basic_widget.emit("app:update_count", {"count": click_count})

def on_reset(data, event_type, label):
    """Reset the counter."""
    print(f"[CALLBACK] on_reset called! data={data}, event_type={event_type}")
    global click_count
    click_count = 0
    basic_widget.emit("app:reset_count", {})

# Show the widget
basic_widget = app.show(
    basic_html,
    title="Basic Toolbar Demo",
    toolbars=[basic_toolbar],
    callbacks={
        "app:click": on_button_click,
        "app:reset": on_reset,
    },
    height=200
)

In [ ]:
# Check widget type - should be PyWryWidget after kernel restart
print(f"Widget type: {type(basic_widget).__module__}.{type(basic_widget).__name__}")

# Check if callbacks are registered
if hasattr(basic_widget, '_handlers'):
    print(f"Registered handlers: {list(basic_widget._handlers.keys())}")
else:
    print("No _handlers attribute - might be InlineWidget")

# Display the widget
basic_widget

---

## 2. Input Components with 2-Way Binding

PyWry provides various input components that automatically emit events when their values change:
- **TextInput** - Text field with debounced input
- **NumberInput** - Numeric input with min/max/step
- **Select** - Dropdown selection
- **SliderInput** - Range slider

In [ ]:
# State for tracking input values
form_state = {
    "name": "World",
    "size": 24,
    "color": "blue",
    "opacity": 100,
}

def update_display():
    """Send current state to update the display."""
    inputs_widget.emit("app:update_display", form_state)

def on_name_change(data, event_type, label):
    """Handle text input change."""
    form_state["name"] = data.get("value", "World")
    update_display()

def on_size_change(data, event_type, label):
    """Handle number input change."""
    form_state["size"] = data.get("value", 24)
    update_display()

def on_color_change(data, event_type, label):
    """Handle select change."""
    form_state["color"] = data.get("value", "blue")
    update_display()

def on_opacity_change(data, event_type, label):
    """Handle slider change."""
    form_state["opacity"] = data.get("value", 100)
    update_display()

# Create toolbar with various input types
inputs_toolbar = Toolbar(
    position="top",
    items=[
        TextInput(
            label="Name:",
            event="form:name",
            value="World",
            placeholder="Enter a name...",
            debounce=300,  # Wait 300ms after typing stops
        ),
        NumberInput(
            label="Size:",
            event="form:size",
            value=24,
            min=8,
            max=72,
            step=2,
        ),
        Select(
            label="Color:",
            event="form:color",
            options=[
                Option(label="Blue", value="blue"),
                Option(label="Red", value="red"),
                Option(label="Green", value="green"),
                Option(label="Purple", value="purple"),
            ],
            selected="blue",
        ),
        SliderInput(
            label="Opacity:",
            event="form:opacity",
            value=100,
            min=0,
            max=100,
            step=5,
            show_value=True,
        ),
    ],
)

# HTML with dynamic greeting
inputs_html = """
<div style="text-align: center;">
    <h1 id="greeting" style="font-size: 24px; color: blue; opacity: 1;">Hello, World!</h1>
    <p style="color: var(--pywry-text-secondary); margin-top: 20px;">
        Try changing the inputs above to update this text
    </p>
</div>

<script>
// Color map for CSS
var colorMap = {
    'blue': '#0078d4',
    'red': '#dc3545',
    'green': '#28a745',
    'purple': '#6f42c1'
};

// Listen for display updates from Python
if (window.pywry && window.pywry.on) {
    window.pywry.on('app:update_display', function(data) {
        var el = document.getElementById('greeting');
        el.textContent = 'Hello, ' + data.name + '!';
        el.style.fontSize = data.size + 'px';
        el.style.color = colorMap[data.color] || data.color;
        el.style.opacity = data.opacity / 100;
    });
}
</script>
"""

# Show the widget
inputs_widget = app.show(
    inputs_html,
    title="Input Components Demo",
    toolbars=[inputs_toolbar],
    callbacks={
        "form:name": on_name_change,
        "form:size": on_size_change,
        "form:color": on_color_change,
        "form:opacity": on_opacity_change,
    },
    height=300
)

---

## 3. Multiple Toolbars with Different Positions

Toolbars can be positioned at:
- **header** - Full-width bar at very top (outside left/right)
- **footer** - Full-width bar at very bottom (outside left/right)
- **top** - Horizontal bar (inside left/right columns)
- **bottom** - Horizontal bar (inside left/right columns)
- **left** - Vertical bar extending from header to footer
- **right** - Vertical bar extending from header to footer
- **inside** - Floating overlay (absolute positioned)

Layout structure:
```
HEADER (full width)
LEFT | TOP / CONTENT / BOTTOM | RIGHT
FOOTER (full width)
```

In [ ]:
def on_position_click(data, event_type, label):
    """Handle position button clicks."""
    position = data.get("position", "unknown")
    positions_widget.emit("app:show_position", {"position": position})

# Create toolbars for each position
top_toolbar = Toolbar(
    position="top",
    items=[Button(label="Top Button", event="pos:click", data={"position": "top"})],
    style="justify-content: center;",
)

bottom_toolbar = Toolbar(
    position="bottom",
    items=[Button(label="Bottom Button", event="pos:click", data={"position": "bottom"})],
    style="justify-content: center;",
)

left_toolbar = Toolbar(
    position="left",
    items=[
        Button(label="L1", event="pos:click", data={"position": "left-1"}, style="width: 100%; "),
        Button(label="L2", event="pos:click", variant="danger", data={"position": "left-2"}, style="width: 100%; "),
    ],
    style="justify-content: center;",
)

inside_toolbar = Toolbar(
    position="inside",
    items=[Button(label="Overlay", event="pos:click", data={"position": "inside"}, variant="outline")],
)

positions_html = """
<div style="height: 100%; width: 100%; display: flex; align-items: center; justify-content: center; text-align: center;">
    <div>
        <h2>Multiple Toolbar Positions</h2>
        <p id="last-click">Click any button...</p>
    </div>
</div>

<script>
if (window.pywry && window.pywry.on) {
    window.pywry.on('app:show_position', function(data) {
        document.getElementById('last-click').textContent = 'Last click: ' + data.position;
    });
}
</script>
"""

positions_widget = app.show(
    positions_html,
    title="Toolbar Positions Demo",
    toolbars=[left_toolbar, top_toolbar, bottom_toolbar, inside_toolbar],
    callbacks={"pos:click": on_position_click},
    height=250
)

---

## 4. Div Components - Labels, Separators & Layout

The `Div` component lets you add non-interactive elements to toolbars:

- **Labels** - Add text like "File:" or "View:" before button groups
- **Separators** - Visual dividers between toolbar sections  
- **Custom HTML** - Any inline content (icons, badges, status indicators)

This creates a more organized, app-like toolbar layout.

In [ ]:
# Track current file
current_file = {"name": "Untitled", "saved": True}

def on_file_action(data, event_type, label):
    """Handle file actions."""
    action = data.get("action", "")
    if action == "new":
        current_file["name"] = "Untitled"
        current_file["saved"] = False
    elif action == "save":
        current_file["saved"] = True
    div_widget.emit("app:update_status", current_file)

def on_view_change(data, event_type, label):
    """Handle view toggle."""
    view = data.get("view", "edit")
    div_widget.emit("app:change_view", {"view": view})

# Toolbar using Div to create logical groups
div_toolbar = Toolbar(
    position="top",
    items=[
        # Group 1: File actions with a label
        Div(
            content="<strong>File:</strong>",
            event="app:file_label",
            style="margin-right: 4px;",
        ),
        Button(label="New", event="app:file", data={"action": "new"}, variant="secondary"),
        Button(label="Save", event="app:file", data={"action": "save"}, variant="neutral"),
        
        # Separator - use a div with border styling
        Div(
            content="",
            event="app:separator",
            style="width: 1px; height: 24px; background: var(--pywry-border-color); margin: 0 12px;",
        ),
        
        # Group 2: View toggle buttons
        Div(
            content="<strong>View:</strong>",
            event="app:view_label",
            style="margin-right: 4px;",
        ),
        Button(label="Edit", event="app:view", data={"view": "edit"}, variant="outline"),
        Button(label="Preview", event="app:view", data={"view": "preview"}, variant="outline"),
    ],
)

div_html = """
<div style="text-align: center;">
    <h2 id="file-name">Untitled</h2>
    <p id="file-status" style="color: var(--pywry-text-secondary); margin: 10px 0;">
        ● Saved
    </p>
    <p id="view-mode" style="color: var(--pywry-accent); margin-top: 20px;">
        Mode: Edit
    </p>
</div>

<script>
if (window.pywry && window.pywry.on) {
    window.pywry.on('app:update_status', function(data) {
        document.getElementById('file-name').textContent = data.name;
        document.getElementById('file-status').textContent = data.saved ? '● Saved' : '○ Unsaved';
        document.getElementById('file-status').style.color = data.saved ? 'var(--pywry-text-secondary)' : '#ef4444';
    });
    window.pywry.on('app:change_view', function(data) {
        document.getElementById('view-mode').textContent = 'Mode: ' + data.view.charAt(0).toUpperCase() + data.view.slice(1);
    });
}
</script>
"""

div_widget = app.show(
    div_html,
    title="Div for Toolbar Layout",
    toolbars=[div_toolbar],
    callbacks={
        "app:file": on_file_action,
        "app:view": on_view_change,
    },
    height=250,
    width="50%",
)

---

## 5. Collapsible & Resizable Toolbars

Toolbars can be made:
- **Collapsible** - Toggle visibility with a collapse button
- **Resizable** - Drag to resize (direction depends on position)

State is automatically persisted in `sessionStorage`.

In [ ]:
# State for tracking sidebar selection
sidebar_state = {"selected": None, "resize_count": 0, "collapsed": False}

def on_toolbar_collapse(data, event_type, label):
    """Handle toolbar collapse/expand events."""
    sidebar_state["collapsed"] = data.get("collapsed", False)
    state_text = "collapsed" if sidebar_state["collapsed"] else "expanded"
    interactive_widget.emit("app:update_state", {"message": f"Sidebar {state_text}"})

def on_toolbar_resize(data, event_type, label):
    """Handle toolbar resize events."""
    sidebar_state["resize_count"] += 1
    width = data.get("width", 0)
    interactive_widget.emit("app:update_state", {"message": f"Sidebar width: {width}px (resized {sidebar_state['resize_count']}x)"})

def on_sidebar_item_click(data, event_type, label):
    """Handle sidebar item click - highlight the selected item."""
    item = data.get("item", "unknown")
    sidebar_state["selected"] = item
    interactive_widget.emit("app:select_item", {"item": item})

def on_action_click(data, event_type, label):
    """Handle action button clicks."""
    action = data.get("action", "unknown")
    interactive_widget.emit("app:show_action", {"action": action})

# Header toolbar
header = Toolbar(
    position="header",
    class_name="pywry-toolbar-header",
    items=[
        Div(
            content="<h2>Interactive Toolbar Features</h2>",
            style="align-items: center; justify-content: center; display: flex; width: 100%;",
        ),
    ]
)

# Bottom toolbar with actions
collapsible_toolbar = Toolbar(
    position="bottom",
    items=[
        Button(label="Show Alert", event="panel:action", data={"action": "alert"}, variant="neutral"),
        Button(label="Change Color", event="panel:action", data={"action": "color"}),
        Select(
            label="Theme:",
            event="panel:theme",
            options=["Default", "Blue", "Green", "Purple"],
            selected="Default",
        ),
    ],
)

# Resizable left sidebar with clickable items
resizable_sidebar = Toolbar(
    position="left",
    resizable=True,
    collapsible=True,
    class_name="sidebar",
    items=[
        Div(
            content="<strong>Navigation</strong>",
            event="sidebar:header",
            style="border-bottom: 1px solid var(--pywry-text-secondary); margin-right: 5px; padding-bottom: 4px;",
        ),
        Button(label="📊 Dashboard", event="sidebar:item", data={"item": "dashboard"}, variant="ghost"),
        Button(label="📁 Files", event="sidebar:item", data={"item": "files"}, variant="ghost"),
        Button(label="⚙️ Settings", event="sidebar:item", data={"item": "settings"}, variant="ghost"),
    ],
)

def on_theme_change(data, event_type, label):
    """Handle theme select change."""
    theme = data.get("value", "Default")
    interactive_widget.emit("app:change_theme", {"theme": theme})

interactive_html = """
<div id="main-content" style="display: flex; flex-direction: column; height: 100%; padding: 12px;">
    <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 8px;">
        <span id="current-page" style="font-size: 18px; font-weight: 600;">📊 Dashboard</span>
        <span id="status-message" style="color: var(--pywry-text-secondary); font-size: 13px;">— Select from sidebar</span>
    </div>
    <div id="action-display" style="flex: 1; background: var(--pywry-bg-secondary); border-radius: 6px; padding: 12px; font-size: 13px; color: var(--pywry-text-secondary);">
        <div style="display: grid; grid-template-columns: repeat(2, 1fr); gap: 4px 16px;">
            <span>• Click sidebar items</span>
            <span>• Drag edge to resize</span>
            <span>• Arrow to collapse</span>
            <span>• Bottom toolbar actions</span>
        </div>
    </div>
</div>

<script>
var themeColors = {
    'Default': 'var(--pywry-bg-secondary)',
    'Blue': 'rgba(0, 120, 212, 0.15)',
    'Green': 'rgba(40, 167, 69, 0.15)',
    'Purple': 'rgba(111, 66, 193, 0.15)'
};

if (window.pywry && window.pywry.on) {
    window.pywry.on('app:select_item', function(data) {
        var icons = {'dashboard': '📊', 'files': '📁', 'settings': '⚙️'};
        var names = {'dashboard': 'Dashboard', 'files': 'Files', 'settings': 'Settings'};
        document.getElementById('current-page').textContent = icons[data.item] + ' ' + names[data.item];
        document.getElementById('status-message').textContent = '— Viewing ' + names[data.item];
    });
    
    window.pywry.on('app:update_state', function(data) {
        document.getElementById('status-message').textContent = '— ' + data.message;
    });
    
    window.pywry.on('app:show_action', function(data) {
        var msg = document.getElementById('status-message');
        if (data.action === 'alert') {
            msg.textContent = '— 🔔 Alert triggered!';
            msg.style.color = '#0078d4';
            setTimeout(function() { msg.style.color = 'var(--pywry-text-secondary)'; }, 1500);
        } else if (data.action === 'color') {
            var colors = ['#0078d4', '#28a745', '#6f42c1', '#dc3545', '#fd7e14'];
            var randomColor = colors[Math.floor(Math.random() * colors.length)];
            document.getElementById('current-page').style.color = randomColor;
            msg.textContent = '— Color changed!';
        }
    });
    
    window.pywry.on('app:change_theme', function(data) {
        var bg = themeColors[data.theme] || themeColors['Default'];
        document.getElementById('action-display').style.background = bg;
        document.getElementById('status-message').textContent = '— Theme: ' + data.theme;
    });
}
</script>
"""

interactive_widget = app.show(
    interactive_html,
    title="Collapsible & Resizable Demo",
    toolbars=[header, collapsible_toolbar, resizable_sidebar],
    callbacks={
        "toolbar:collapse": on_toolbar_collapse,
        "toolbar:expand": on_toolbar_collapse,
        "toolbar:resize": on_toolbar_resize,
        "sidebar:item": on_sidebar_item_click,
        "panel:action": on_action_click,
        "panel:theme": on_theme_change,
    },
    height=325,
    width="75%",
)

---

## 6. Advanced Example: Interactive Dashboard

This example combines multiple toolbar features into a cohesive interactive dashboard:
- Real-time filtering
- Multiple input types
- Dynamic content updates
- Nested Div layouts

In [ ]:
# Dashboard state
dashboard_state = {
    "search": "",
    "category": "all",
    "price_min": 0,
    "price_max": 1000,
    "sort": "name",
    "items": [
        {"name": "Laptop Pro", "category": "electronics", "price": 999},
        {"name": "Wireless Mouse", "category": "electronics", "price": 29},
        {"name": "Desk Chair", "category": "furniture", "price": 299},
        {"name": "Standing Desk", "category": "furniture", "price": 499},
        {"name": "Notebook Set", "category": "office", "price": 15},
        {"name": "Monitor 27\"", "category": "electronics", "price": 399},
        {"name": "Bookshelf", "category": "furniture", "price": 149},
        {"name": "Pen Holder", "category": "office", "price": 12},
    ],
}

def filter_items():
    """Filter items based on current state."""
    items = dashboard_state["items"]
    search = dashboard_state["search"].lower()
    category = dashboard_state["category"]
    price_min = dashboard_state["price_min"]
    price_max = dashboard_state["price_max"]
    
    filtered = [
        item for item in items
        if (search == "" or search in item["name"].lower())
        and (category == "all" or item["category"] == category)
        and (price_min <= item["price"] <= price_max)
    ]
    
    # Sort
    sort_key = dashboard_state["sort"]
    filtered.sort(key=lambda x: x.get(sort_key, ""))
    
    return filtered

def update_dashboard():
    """Send filtered results to frontend."""
    filtered = filter_items()
    dashboard_widget.emit("dashboard:update", {
        "items": filtered,
        "total": len(dashboard_state["items"]),
        "filtered": len(filtered),
    })

# Handlers
def on_search(data, event_type, label):
    dashboard_state["search"] = data.get("value", "")
    update_dashboard()

def on_category(data, event_type, label):
    dashboard_state["category"] = data.get("value", "all")
    update_dashboard()

def on_price_range(data, event_type, label):
    dashboard_state["price_min"] = data.get("start", 0)
    dashboard_state["price_max"] = data.get("end", 1000)
    update_dashboard()

def on_sort(data, event_type, label):
    dashboard_state["sort"] = data.get("value", "name")
    update_dashboard()

def on_reset_filters(data, event_type, label):
    """Reset all filters to defaults."""
    dashboard_state["search"] = ""
    dashboard_state["category"] = "all"
    dashboard_state["price_min"] = 0
    dashboard_state["price_max"] = 1000
    dashboard_state["sort"] = "name"
    update_dashboard()

# Header with title
header_toolbar = Toolbar(
    position="header",
    items=[
        Div(
            content="<strong style='font-size: 16px;'>📦 Product Dashboard</strong>",
            style="padding: 4px 8px;",
        ),
    ],
)

# Top toolbar with search and category filter
search_toolbar = Toolbar(
    position="top",
    items=[
        TextInput(
            label="Search:",
            event="dash:search",
            placeholder="Filter by name...",
            debounce=200,
        ),
        Select(
            label="Category:",
            event="dash:category",
            options=[
                Option(label="All", value="all"),
                Option(label="Electronics", value="electronics"),
                Option(label="Furniture", value="furniture"),
                Option(label="Office", value="office"),
            ],
            selected="all",
        ),
        Select(
            label="Sort:",
            event="dash:sort",
            options=[
                Option(label="Name", value="name"),
                Option(label="Price", value="price"),
                Option(label="Category", value="category"),
            ],
            selected="name",
        ),
    ],
)

# Bottom toolbar with RangeInput for price filtering
price_toolbar = Toolbar(
    position="bottom",
    items=[
        RangeInput(
            label="Price Range:",
            event="dash:price",
            min=0,
            max=1000,
            step=10,
            start=0,
            end=1000,
            show_value=True,
        ),
        Button(label="Reset Filters", event="dash:reset", variant="secondary"),
    ],
)

# Footer with count
footer_toolbar = Toolbar(
    position="footer",
    items=[
        Div(
            content="<span id='result-count'>Showing 8 of 8 items</span>",
            style="padding: 4px 8px; color: var(--pywry-text-secondary);",
        ),
    ],
)

dashboard_html = """
<div id="items-container" style="display: grid; grid-template-columns: repeat(auto-fill, minmax(180px, 1fr)); gap: 12px; padding: 8px;">
</div>

<script>
function renderItems(items) {
    var container = document.getElementById('items-container');
    container.innerHTML = items.map(function(item) {
        var categoryColors = {
            'electronics': '#0078d4',
            'furniture': '#28a745',
            'office': '#6f42c1'
        };
        var color = categoryColors[item.category] || '#666';
        return '<div style="background: var(--pywry-bg-secondary); border-radius: 8px; padding: 12px; border: 1px solid var(--pywry-border-color);">' +
            '<h3 style="margin: 0 0 8px 0; font-size: 14px;">' + item.name + '</h3>' +
            '<span style="background: ' + color + '; color: white; padding: 2px 6px; border-radius: 4px; font-size: 11px;">' + item.category + '</span>' +
            '<p style="margin: 8px 0 0 0; font-size: 18px; font-weight: bold; color: var(--pywry-accent);">$' + item.price + '</p>' +
        '</div>';
    }).join('');
    
    if (items.length === 0) {
        container.innerHTML = '<div style="grid-column: 1/-1; text-align: center; padding: 40px; color: var(--pywry-text-secondary);">No items match your filters</div>';
    }
}

// Initial render
var initialItems = %INITIAL_ITEMS%;
renderItems(initialItems);

// Listen for updates from Python
if (window.pywry && window.pywry.on) {
    window.pywry.on('dashboard:update', function(data) {
        renderItems(data.items);
        var countEl = document.getElementById('result-count');
        if (countEl) {
            countEl.textContent = 'Showing ' + data.filtered + ' of ' + data.total + ' items';
        }
    });
}
</script>
"""

# Inject initial items
import json
dashboard_html_final = dashboard_html.replace('%INITIAL_ITEMS%', json.dumps(dashboard_state["items"]))

dashboard_widget = app.show(
    dashboard_html_final,
    title="Product Dashboard",
    toolbars=[header_toolbar, search_toolbar, price_toolbar, footer_toolbar],
    callbacks={
        "dash:search": on_search,
        "dash:category": on_category,
        "dash:price": on_price_range,
        "dash:sort": on_sort,
        "dash:reset": on_reset_filters,
    },
    height=400
)

---

## 7. All Toolbar Components - Zero JS/HTML

This example shows every toolbar component using **only Python** - no custom JavaScript or HTML needed.
The entire UI is built declaratively with PyWry's toolbar system.

### Built-in System Events Demonstrated

| Event | Usage | Description |
|-------|-------|-------------|
| `pywry:update_theme` | Theme toggle (☀️) | Switches between dark/light mode |
| `pywry:inject-css` | Accent color select | Dynamically injects CSS to change accent color |
| `pywry:set_style` | Title size & Labels selects | Updates inline styles by id or CSS selector |
| `pywry:set_content` | Footer display | Updates element innerHTML from Python |

In [ ]:
# State to display current values
component_values = {}
current_theme = "dark"  # Track current theme

def make_handler(name):
    """Create a handler that updates the display with the component's value."""
    def handler(data, event_type, label):
        # Extract the relevant value(s) from the event data
        if "value" in data:
            component_values[name] = data["value"]
        elif "values" in data:
            component_values[name] = data["values"]
        elif "start" in data and "end" in data:
            component_values[name] = f"{data['start']} - {data['end']}"
        elif "btn" in data:
            component_values[name] = data["btn"]
        else:
            component_values[name] = "clicked"
        
        # Build display text for footer using built-in pywry:set_content
        parts = [f"<strong>{k}:</strong> {v}" for k, v in component_values.items()]
        components_widget.emit("pywry:set_content", {
            "id": "values-display",
            "html": " | ".join(parts)
        })
    return handler

def on_theme_toggle(data, event_type, label):
    """Toggle between dark and light mode using built-in pywry event."""
    global current_theme
    current_theme = "light" if current_theme == "dark" else "dark"
    components_widget.emit("pywry:update_theme", {"theme": current_theme})

def on_title_size(data, event_type, label):
    """Change the title size using built-in pywry:set_style event."""
    sizes = {"sm": "16px", "md": "20px", "lg": "26px"}
    size = data.get("value", "md")
    # Use built-in pywry:set_style event to update element styles
    components_widget.emit("pywry:set_style", {
        "id": "demo-title",
        "styles": {"fontSize": sizes.get(size, "20px")}
    })

def on_label_style(data, event_type, label):
    """Change all component label styles using built-in pywry:set_style event."""
    style_map = {
        "normal": {"fontWeight": "400", "fontStyle": "normal"},
        "semi": {"fontWeight": "500", "fontStyle": "normal"},
        "bold": {"fontWeight": "700", "fontStyle": "normal"},
        "italic": {"fontWeight": "400", "fontStyle": "italic"},
    }
    style = data.get("value", "normal")
    # Use built-in pywry:set_style event to update all labels
    components_widget.emit("pywry:set_style", {
        "selector": ".pywry-input-label",
        "styles": style_map.get(style, style_map["normal"])
    })

def on_accent_color(data, event_type, label):
    """Change the accent color using built-in pywry:inject-css event."""
    colors = {
        "blue": "#0078d4",
        "green": "#28a745",
        "purple": "#6f42c1",
        "orange": "#fd7e14",
        "pink": "#e91e63",
    }
    color = data.get("value", "blue")
    accent = colors.get(color, colors["blue"])
    
    # Use built-in pywry:inject-css to dynamically inject CSS
    # The 'id' allows replacing the same style block on subsequent calls
    components_widget.emit("pywry:inject-css", {
        "id": "custom-accent-color",
        "css": f"""
            :root {{
                --pywry-accent: {accent} !important;
                --pywry-accent-hover: {accent}dd !important;
            }}
            .pywry-btn-neutral {{
                background: {accent} !important;
            }}
            .pywry-btn-neutral:hover {{
                background: {accent}dd !important;
            }}
        """
    })

# Header toolbar with title and theme toggle
components_header = Toolbar(
    position="header",
    items=[
        Div(
            content="<h3 id='demo-title' style='margin: 0;'>🧩 All Toolbar Components</h3>",
            style="flex: 1;",
        ),
        Select(
            label="Accent:",
            event="demo:accent",
            options=[
                Option(label="Blue", value="blue"),
                Option(label="Green", value="green"),
                Option(label="Purple", value="purple"),
                Option(label="Orange", value="orange"),
                Option(label="Pink", value="pink"),
            ],
            selected="blue",
        ),
        Select(
            label="Title:",
            event="demo:title_size",
            options=[Option(label="SM", value="sm"), Option(label="MD", value="md"), Option(label="LG", value="lg")],
            selected="md",
        ),
        Select(
            label="Labels:",
            event="demo:label_style",
            options=[Option(label="Normal", value="normal"), Option(label="Semi", value="semi"), Option(label="Bold", value="bold"), Option(label="Italic", value="italic")],
            selected="normal",
        ),
        Button(label="☀️", event="demo:theme", variant="ghost", component_id="theme-toggle-btn"),
    ],
)

# Top toolbar with text, number, and date inputs
inputs_row = Toolbar(
    position="top",
    items=[
        TextInput(
            label="Text:",
            event="demo:text",
            value="Hello",
            placeholder="Type here...",
        ),
        NumberInput(
            label="Number:",
            event="demo:number",
            value=42,
            min=0,
            max=100,
            step=1,
        ),
        DateInput(
            label="Date:",
            event="demo:date",
            value="2026-01-13",
        ),
    ],
)

# Second row with select, multi-select
selects_row = Toolbar(
    position="top",
    items=[
        Select(
            label="Select:",
            event="demo:select",
            options=[
                Option(label="Option A", value="a"),
                Option(label="Option B", value="b"),
                Option(label="Option C", value="c"),
            ],
            selected="a",
        ),
        MultiSelect(
            label="Multi:",
            event="demo:multi",
            options=[
                Option(label="Red", value="red"),
                Option(label="Green", value="green"),
                Option(label="Blue", value="blue"),
            ],
            selected=["red"],
        ),
    ],
)

# Third row with sliders and range
sliders_row = Toolbar(
    position="top",
    items=[
        SliderInput(
            label="Slider:",
            event="demo:slider",
            value=50,
            min=0,
            max=100,
            step=5,
            show_value=True,
        ),
        RangeInput(
            label="Range:",
            event="demo:range",
            min=0,
            max=100,
            start=20,
            end=80,
            show_value=True,
        ),
    ],
)

# Fourth row with toggle, checkbox, and horizontal radio
booleans_row = Toolbar(
    position="top",
    items=[
        Toggle(label="Toggle:", event="demo:toggle", value=True),
        Div(content="<span class='pywry-input-label'>Check:</span>", style="margin-right: 4px;"),
        Checkbox(label="", event="demo:checkbox", value=False),
        RadioGroup(
            label="Radio:",
            event="demo:radio",
            options=[Option(label="A", value="a"), Option(label="B", value="b"), Option(label="C", value="c")],
            selected="a",
            direction="horizontal",
        ),
    ],
)

# Fifth row with TabGroups
tabs_row = Toolbar(
    position="top",
    items=[
        TabGroup(
            label="View:",
            event="demo:tabs",
            options=[
                Option(label="Table", value="table"),
                Option(label="Chart", value="chart"),
                Option(label="Map", value="map"),
            ],
            selected="table",
        ),
        TabGroup(
            label="Size:",
            event="demo:tabsize",
            options=["SM", "MD", "LG"],
            selected="MD",
            size="sm",
        ),
    ],
)

# Right sidebar with vertical radio group
right_sidebar = Toolbar(
    position="right",
    style="padding: 8px 12px;",
    items=[
        Div(content="<span class='pywry-input-label'>Priority</span>", style="margin-bottom: 4px;"),
        RadioGroup(
            event="demo:priority",
            options=[
                Option(label="Low", value="low"),
                Option(label="Medium", value="med"),
                Option(label="High", value="high"),
            ],
            selected="med",
            direction="vertical",
        ),
    ],
    collapsible=True,
)

# Bottom toolbar with buttons (all variants)
buttons_row = Toolbar(
    position="top",
    items=[
        Div(content="<span class='pywry-input-label'>Variants:</span>", style="margin-right: 4px;"),
        Button(label="Primary", event="demo:btn", data={"btn": "primary"}, variant="primary"),
        Button(label="Secondary", event="demo:btn", data={"btn": "secondary"}, variant="secondary"),
        Button(label="Neutral", event="demo:btn", data={"btn": "neutral"}, variant="neutral"),
        Button(label="Ghost", event="demo:btn", data={"btn": "ghost"}, variant="ghost"),
        Button(label="Outline", event="demo:btn", data={"btn": "outline"}, variant="outline"),
        Button(label="Danger", event="demo:btn", data={"btn": "danger"}, variant="danger"),
        Button(label="Warning", event="demo:btn", data={"btn": "warning"}, variant="warning"),
        Button(label="⚙", event="demo:btn", data={"btn": "icon"}, variant="icon"),
    ],
)

# Size variants row
sizes_row = Toolbar(
    position="top",
    items=[
        Div(content="<span class='pywry-input-label'>Sizes</span>", style="margin-right: 4px;"),
        Button(label="XS", event="demo:btn", data={"btn": "xs"}, variant="neutral", size="xs"),
        Button(label="SM", event="demo:btn", data={"btn": "sm"}, variant="neutral", size="sm"),
        Button(label="Default", event="demo:btn", data={"btn": "default"}, variant="neutral"),
        Button(label="LG", event="demo:btn", data={"btn": "lg"}, variant="neutral", size="lg"),
        Button(label="XL", event="demo:btn", data={"btn": "xl"}, variant="neutral", size="xl"),
    ],
)

# Footer with live status display
components_footer = Toolbar(
    position="footer",
    items=[
        Div(
            content="<span id='values-display'><em>Interact with components above...</em></span>",
            style="color: var(--pywry-text-secondary); width: 100%; text-align: center;",
        ),
    ],
)

# No custom JS needed - all interactions use built-in pywry events!
components_html = ""

components_widget = app.show(
    components_html,
    title="All Components Demo",
    toolbars=[
        components_header,
        inputs_row,
        selects_row,
        sliders_row,
        booleans_row,
        tabs_row,
        buttons_row,
        sizes_row,
        right_sidebar,
        components_footer,
    ],
    callbacks={
        "demo:text": make_handler("Text"),
        "demo:number": make_handler("Number"),
        "demo:date": make_handler("Date"),
        "demo:select": make_handler("Select"),
        "demo:multi": make_handler("Multi"),
        "demo:slider": make_handler("Slider"),
        "demo:range": make_handler("Range"),
        "demo:toggle": make_handler("Toggle"),
        "demo:checkbox": make_handler("Checkbox"),
        "demo:radio": make_handler("Radio"),
        "demo:tabs": make_handler("Tabs"),
        "demo:tabsize": make_handler("TabSize"),
        "demo:priority": make_handler("Priority"),
        "demo:btn": make_handler("Button"),
        "demo:theme": on_theme_toggle,
        "demo:title_size": on_title_size,
        "demo:label_style": on_label_style,
        "demo:accent": on_accent_color,
    },
    height=375
)

---

## Summary

### Toolbar Components

| Component | Event Data | Description |
|-----------|------------|-------------|
| `Button` | `{componentId, ...data}` | Click triggers event |
| `TextInput` | `{componentId, value}` | Debounced text input |
| `NumberInput` | `{componentId, value}` | Numeric with min/max/step |
| `Select` | `{componentId, value}` | Dropdown selection |
| `MultiSelect` | `{componentId, values}` | Checkbox group dropdown |
| `SliderInput` | `{componentId, value}` | Single value slider |
| `RangeInput` | `{componentId, start, end}` | Dual-handle range slider |
| `DateInput` | `{componentId, value}` | Date picker |
| `Toggle` | `{componentId, value}` | Boolean switch (on/off) |
| `Checkbox` | `{componentId, value}` | Boolean checkbox |
| `RadioGroup` | `{componentId, value}` | Radio buttons (single selection) |
| `TabGroup` | `{componentId, value}` | Tab-style selection (single selection) |
| `Div` | (custom) | Container for HTML/children |

### Boolean Components

| Component | Description |
|-----------|-------------|
| `Toggle` | Sliding switch for on/off states |
| `Checkbox` | Standard checkbox with label |
| `RadioGroup` | Radio buttons with `direction="horizontal"` or `"vertical"` |

### Selection Components

| Component | Description |
|-----------|-------------|
| `RadioGroup` | Radio buttons - use for few options in form context |
| `TabGroup` | Tab-style buttons - use for view/mode switching |

### Button Variants

| Variant | Description |
|---------|-------------|
| `primary` | Theme-aware (dark in light mode, light in dark mode) |
| `secondary` | Subtle background, theme-aware |
| `neutral` | **Always blue accent** - use for primary actions |
| `ghost` | Transparent background |
| `outline` | Bordered, transparent background |
| `danger` | Red - for destructive actions |
| `warning` | Orange - for caution/warning actions |
| `icon` | Ghost style, square aspect ratio for icon-only buttons |

### Button Sizes

| Size | Description |
|------|-------------|
| `xs` | Extra small (11px font, 2px/6px padding) |
| `sm` | Small (12px font, 4px/8px padding) |
| (default) | Normal size (13px font, 4px/8px padding) |
| `lg` | Large (15px font, 10px/20px padding) |
| `xl` | Extra large (16px font, 14px/28px padding) |

### Toolbar Options

| Option | Type | Description |
|--------|------|-------------|
| `position` | `str` | "top", "bottom", "left", "right", "inside" |
| `collapsible` | `bool` | Add collapse/expand toggle |
| `resizable` | `bool` | Enable drag-to-resize |
| `class_name` | `str` | Custom CSS class |
| `script` | `str` | Custom JavaScript |

### 2-Way Communication Pattern

```python
# Python → JavaScript
widget.emit("event:name", {"key": "value"})

# JavaScript → Python (automatic via toolbar events)
callbacks={"event:name": handler_function}
```